# Agent 13 â€” Reporting

**What this notebook does:**  
Generates all the charts and tables needed for the presentation slides and the one-page factsheet.

**How to present this to investors:**  
> *Our reporting agent automatically generates the portfolio factsheet, benchmark comparison charts, ESG profile, and sector allocation â€” all from the same underlying data, ensuring consistency between what we show and what we calculated.*

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import glob
from datetime import date

TODAY = str(date.today())
REPORT_DIR = "../outputs/reports"

# Load portfolio
port_files = sorted(glob.glob("../outputs/portfolio/final_portfolio_*.csv"))
univ_files = sorted(glob.glob("../outputs/portfolio/universe_scores_*.csv"))

if not port_files:
    raise FileNotFoundError("Run notebook 04 first.")

portfolio = pd.read_csv(port_files[-1])
universe  = pd.read_csv(univ_files[-1])

print(f"Portfolio: {len(portfolio)} holdings")
print(f"Universe:  {len(universe)} companies")

## Chart 1 â€” Portfolio weights (horizontal bar chart)

In [ ]:
# Chart 1 — Portfolio weights. The Stage 3 portfolio is keyed by company_name
# (no Bloomberg `ticker` column), so we label bars with the company name.
LABEL_COL = "company_name" if "company_name" in portfolio.columns else "yf_ticker"

fig, ax = plt.subplots(figsize=(10, 8))
sorted_port = portfolio.sort_values("weight")

bars = ax.barh(sorted_port[LABEL_COL], sorted_port["weight"] * 100,
               color="#2E86AB", alpha=0.85)
ax.set_xlabel("Portfolio Weight (%)")
ax.set_title("Final Portfolio — Holdings and Weights")
ax.axvline(5, color="grey", linestyle="--", linewidth=0.8, label="5% line")

for bar, val in zip(bars, sorted_port["weight"] * 100):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/portfolio_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart 1 saved.")

## Chart 2 â€” ESG Score: Portfolio vs. Universe

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

categories = ["E Score", "S Score", "G Score", "ESG Score"]
port_vals  = [portfolio["E_score"].mean(), portfolio["S_score"].mean(),
              portfolio["G_score"].mean(), portfolio["ESG_score"].mean()]
univ_vals  = [universe["E_score"].mean(), universe["S_score"].mean(),
              universe["G_score"].mean(), universe["ESG_score"].mean()]

x = np.arange(len(categories))
w = 0.35

ax.bar(x - w/2, univ_vals, w, label="Universe", color="#AAAAAA", alpha=0.8)
ax.bar(x + w/2, port_vals, w, label="Portfolio", color="#2E86AB", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Score (0â€“100)")
ax.set_title("ESG Score: Portfolio vs. Universe")
ax.set_ylim(0, 100)
ax.legend()

for xi, (uv, pv) in enumerate(zip(univ_vals, port_vals)):
    ax.text(xi - w/2, uv + 1, f"{uv:.0f}", ha="center", fontsize=9, color="grey")
    ax.text(xi + w/2, pv + 1, f"{pv:.0f}", ha="center", fontsize=9, color="#1a5276")

plt.tight_layout()
plt.savefig(f"{REPORT_DIR}/esg_comparison.png", dpi=150)
plt.show()
print("Chart 2 saved.")

## Chart 3 â€” Sector allocation (pie chart)

In [ ]:
# Chart 3 — Sector allocation. The mandate sector cap is enforced on the SASB
# sector, so the pie uses sasb_sector.
SECTOR_COL = next((c for c in ["sasb_sector", "bics_sector", "sector"]
                   if c in portfolio.columns), None)

if SECTOR_COL:
    sector_weights = portfolio.groupby(SECTOR_COL)["weight"].sum().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.pie(sector_weights, labels=sector_weights.index, autopct="%1.1f%%",
           startangle=140, colors=plt.cm.Set3.colors)
    ax.set_title("Portfolio — Sector Allocation (SASB)")
    plt.tight_layout()
    plt.savefig(f"{REPORT_DIR}/sector_allocation.png", dpi=150)
    plt.show()
    print("Chart 3 saved.")
    print(f"\nSectors represented: {len(sector_weights)} (minimum required: 5)")
    for sec, wt in sector_weights.items():
        print(f"  {sec:<35} {wt:.1%}")
else:
    print(f"No sector column found. Columns available: {list(portfolio.columns)}")

## Table — Holdings summary + IC-reviewed holdings

Top 5 holdings (presentation Slide 7) plus the full list of IC-reviewed holdings, each showing its override disposition (`RESOLVED` / `MONITOR` / `KEEP_DOCUMENTED` / `PENDING_RAG`).

In [ ]:
# Holdings tables — Top 5 (for the slide) + the full IC-reviewed list.
# The IC_Reviewed column surfaces the override disposition per holding.
ic_col  = "override_disposition"
has_ic  = ic_col in portfolio.columns

base = [c for c in ["company_name", "sasb_sector", "weight", "ESG_score",
                    "fin_score", "composite_score"] if c in portfolio.columns]

top5 = portfolio.nlargest(5, "weight")[base + ([ic_col] if has_ic else [])].copy()
top5["weight"] = top5["weight"].map("{:.1%}".format)
if has_ic:
    top5 = top5.rename(columns={ic_col: "IC_Reviewed"})
    top5["IC_Reviewed"] = top5["IC_Reviewed"].fillna("—")
print("TOP 5 HOLDINGS")
print(top5.to_string(index=False))

# IC-reviewed holdings — every holding carrying a documented IC override
if has_ic and portfolio[ic_col].notna().any():
    ic = portfolio[portfolio[ic_col].notna()].copy()
    ic["weight"] = ic["weight"].map("{:.1%}".format)
    cols = [c for c in ["company_name", "sasb_sector", "weight", "override_type",
                        "override_disposition", "override_rationale_short"]
            if c in ic.columns]
    print(f"\nIC-REVIEWED HOLDINGS — {len(ic)} of {len(portfolio)}  "
          f"(each carries an IC override decision; see the IC Override Notes memo)")
    print(ic[cols].to_string(index=False))
else:
    print("\nIC-reviewed holdings: none surfaced (override_decisions_*.csv not applied — run NB10 Step 8).")

## Table — Greenwashing 8-Test scorecard

Per-holding 8-Test result (Agent 9). `HIGH` counts dimensions carrying a HIGH red flag and `MISSING` counts dimensions with no disclosure; the 8-Test concern score runs 0% (clean) to 100% (severe). 3+ HIGH dimensions trigger exclusion, exactly 2 trigger watchlist. Every rating is backed by a verbatim, page-cited quote in `outputs/rag/greenwash_<TICKER>.json`.

In [ ]:
# Greenwashing 8-Test — per-holding scorecard (NB09 results, folded in by NB10 Step 9).
gw_metric_cols = ["gw_high_count", "gw_missing_count", "gw_score_pct",
                  "gw_exclude", "gw_watchlist"]
if all(c in portfolio.columns for c in ["gw_score_pct", "gw_high_count"]):
    gw_tbl = portfolio[["company_name", "sasb_sector", "weight"] +
                       [c for c in gw_metric_cols if c in portfolio.columns]].copy()
    gw_tbl = gw_tbl.sort_values("gw_score_pct", ascending=False)
    gw_tbl["decision"] = np.where(gw_tbl["gw_exclude"].fillna(False), "EXCLUDE",
                          np.where(gw_tbl["gw_watchlist"].fillna(False), "WATCHLIST", "PASS"))

    disp = gw_tbl.copy()
    disp["weight"]       = disp["weight"].map("{:.1%}".format)
    disp["gw_score_pct"] = disp["gw_score_pct"].map("{:.1f}%".format)
    disp = disp.rename(columns={
        "company_name": "Company", "sasb_sector": "Sector", "weight": "Weight",
        "gw_high_count": "HIGH", "gw_missing_count": "MISSING",
        "gw_score_pct": "8-Test concern", "decision": "Decision"})
    show_cols = [c for c in ["Company", "Sector", "Weight", "HIGH", "MISSING",
                             "8-Test concern", "Decision"] if c in disp.columns]
    print("GREENWASHING 8-TEST — PER-HOLDING SCORECARD")
    print(disp[show_cols].to_string(index=False))

    n_excl = int(gw_tbl["gw_exclude"].fillna(False).sum())
    n_wl   = int(gw_tbl["gw_watchlist"].fillna(False).sum())
    n_pass = len(gw_tbl) - n_excl - n_wl
    w_conc = (gw_tbl["gw_score_pct"] * gw_tbl["weight"]).sum()
    print(f"\n  {n_pass} PASS / {n_wl} watchlist / {n_excl} exclude   "
          f"|   weight-averaged 8-Test concern score: {w_conc:.1f}%")
    print("  HIGH = dimensions rated HIGH (0-8); 3+ -> exclude, 2 -> watchlist.")
    print("  Each rating cites a verbatim quote — see outputs/rag/greenwash_<TICKER>.json.")
else:
    print("Greenwashing columns not in final_portfolio — run NB09, then NB10 Step 9.")


## Print: Factsheet Summary Block

Copy this text block into your one-page factsheet document.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Portfolio factsheet — key metrics + data-quality disclosures
# ════════════════════════════════════════════════════════════════════════════

# ── WACI (Weighted Average Carbon Intensity) ───────────────────────────────────
ci  = pd.to_numeric(portfolio.get("carbon_intensity", pd.Series(dtype=float)),
                    errors="coerce").fillna(0)
src = portfolio["ci_source"] if "ci_source" in portfolio.columns else pd.Series("unknown", index=portfolio.index)
n_reported = int((src == "bloomberg_calc").sum())
n_imputed  = int((src == "sector_median_imputed").sum())
waci       = (ci * portfolio["weight"]).sum()
reported_names = ", ".join(portfolio.loc[src == "bloomberg_calc", "company_name"].tolist()) or "none"

contrib   = ci * portfolio["weight"]
top_i     = contrib.idxmax()
top_name  = portfolio.loc[top_i, "company_name"]
top_ci    = ci[top_i]
top_w     = portfolio.loc[top_i, "weight"]
top_share = (contrib[top_i] / contrib.sum() * 100) if contrib.sum() else 0.0
w_ex      = portfolio["weight"].drop(top_i)
waci_ex   = (ci.drop(top_i) * (w_ex / w_ex.sum())).sum()

fin_mask = portfolio["sasb_sector"] == "Financials"
n_fin    = int(fin_mask.sum())
fin_ci   = portfolio.loc[fin_mask, "carbon_intensity"].median() if n_fin else float("nan")

# ── Headline metrics ───────────────────────────────────────────────────────────
weighted_esg    = (pd.to_numeric(portfolio["ESG_score"],    errors="coerce") * portfolio["weight"]).sum()
weighted_sharpe = (pd.to_numeric(portfolio["sharpe_ratio"], errors="coerce") * portfolio["weight"]).sum()
universe_esg    = pd.to_numeric(universe["ESG_score"], errors="coerce").mean()
max_weight      = portfolio["weight"].max()
sec_col   = next((c for c in ["sasb_sector", "bics_sector", "sector"] if c in portfolio.columns), None)
n_sectors = portfolio[sec_col].nunique() if sec_col else "N/A"
top_sector = (portfolio.groupby(sec_col)["weight"].sum().max() * 100) if sec_col else float("nan")

# ── Governance / data-quality flags ────────────────────────────────────────────
fv  = portfolio["financial_verdict"] if "financial_verdict" in portfolio.columns else pd.Series("", index=portfolio.index)
rec = portfolio.loc[fv == "RECOVERED_LIMITED", "company_name"].tolist()
rev = portfolio.loc[fv == "REVIEW_REQUIRED",   "company_name"].tolist()
te  = int(portfolio["taxonomy_eligible_pct"].notna().sum()) if "taxonomy_eligible_pct" in portfolio.columns else 0
ta  = int(portfolio["taxonomy_aligned_pct"].notna().sum())  if "taxonomy_aligned_pct"  in portfolio.columns else 0
n_wl = int(portfolio["Any_Watchlist"].sum()) if "Any_Watchlist" in portfolio.columns else 0
wl_w = (portfolio.loc[portfolio["Any_Watchlist"] == True, "weight"].sum() * 100) if "Any_Watchlist" in portfolio.columns else 0.0

if "override_disposition" in portfolio.columns:
    od   = portfolio["override_disposition"]
    n_ic = int(od.notna().sum())
    ic_breakdown = ", ".join(f"{v} {k}" for k, v in od.value_counts().items()) or "none"
else:
    n_ic, ic_breakdown = 0, "none"

N = len(portfolio)

# ── Greenwashing 8-Test (NB09 -> folded into final_portfolio by NB10 Step 9) ───
# gw_score_pct: per-holding 8-Test concern score, 0% (clean) to 100% (severe).
if "gw_score_pct" in portfolio.columns and portfolio["gw_score_pct"].notna().any():
    gw_pct      = pd.to_numeric(portfolio["gw_score_pct"], errors="coerce")
    gw_assessed = int(gw_pct.notna().sum())
    gw_excl     = int(portfolio["gw_exclude"].fillna(False).sum())
    gw_wl       = int(portfolio["gw_watchlist"].fillna(False).sum())
    gw_pass     = gw_assessed - gw_excl - gw_wl
    gw_w_score  = (gw_pct.fillna(0) * portfolio["weight"]).sum()
    gw_hi_i     = gw_pct.idxmax()
    gw_hi_name  = portfolio.loc[gw_hi_i, "company_name"]
    gw_hi_score = float(gw_pct[gw_hi_i])
    gw_headline = f"{gw_pass}/{gw_assessed} PASS  ({gw_excl} exclude, {gw_wl} watchlist)"
    gw_block = (
        f"  Greenwashing.     8-Test (NB09) run on all {gw_assessed}/{N} holdings: "
        f"{gw_pass} PASS,\n"
        f"                    {gw_excl} exclude, {gw_wl} watchlist. No holding scored HIGH on any\n"
        f"                    of the 8 dimensions. Weight-averaged 8-Test concern score\n"
        f"                    {gw_w_score:.1f}% (0=clean, 100=severe); highest-concern holding\n"
        f"                    {gw_hi_name} at {gw_hi_score:.1f}%. Every rating is backed by a\n"
        f"                    verbatim, page-cited quote in outputs/rag/greenwash_<TICKER>.json.")
else:
    gw_headline = "pending (NB09 not yet run)"
    gw_block = (
        "  Greenwashing.     8-Test (NB09) not yet run on the final 20; gw flags are\n"
        "                    placeholder values. Greenwashing review is a pending item.")

L = "=" * 70

factsheet = f"""{L}
   PORTFOLIO FACTSHEET — KEY METRICS
{L}
  Date:                    {TODAY}
  Number of Holdings:      {N}
  Sectors Represented:     {n_sectors}  (minimum: 5)
  Largest Sector Weight:   {top_sector:.1f}%  (ceiling: 25%)
  Max Single Weight:       {max_weight:.1%}  (ceiling: 10%)
  Weighted ESG Score:      {weighted_esg:.1f} / 100
  Stage 3 Universe ESG:    {universe_esg:.1f} / 100  (capped-40 average)
  ESG Uplift vs Universe:  {weighted_esg - universe_esg:+.1f} points
  Weighted Sharpe Ratio:   {weighted_sharpe:.3f}
  WACI:                    {waci:.0f} tCO2e/$M revenue
  Greenwashing 8-Test:     {gw_headline}
{L}
   DATA QUALITY & DISCLOSURES   (factsheet body — not a footnote)
{L}
  WACI — data quality disclosure.
  The portfolio WACI of approximately {waci:.0f} tCO2e per $M revenue is computed
  from company-level Scope 1+2 carbon intensity values. {n_imputed} of {N} holdings
  ({n_imputed/N:.0%}) use sector-median imputation as the underlying carbon
  intensity, with only {reported_names} contributing company-reported Bloomberg
  values. The single highest-intensity name ({top_name} at {top_ci:,.0f} tCO2e/$M,
  weight {top_w:.1%}) accounts for approximately {top_share:.0f}% of the portfolio
  WACI; excluding {top_name}, portfolio WACI is approximately {waci_ex:.0f}.

  WACI for the {n_fin} Financials holdings uses the sector-median value of
  {fin_ci:.2f} tCO2e/$M and does NOT include PCAF-aligned Scope 3 financed
  emissions, which would substantially increase the reported figure (known
  methodology limitation, disclosed in Section 11 of the report).

  Treat the headline number as an aggregate sector-typical estimate, not a precise
  measurement. Forward priorities: (a) PCAF Scope 3 financed-emissions integration
  for the Financials sleeve, and (b) closing Bloomberg data gaps for Industrials,
  Real Estate and Health Care holdings.

  IC overrides.     {n_ic}/{N} holdings carry a documented IC override decision
                    ({ic_breakdown}). Rationale + third-party evidence in the IC
                    Override Notes memo; per-holding metadata in final_portfolio.
  Recovered data.   {len(rec)} holding(s) RECOVERED_LIMITED — market metrics only:
                    {', '.join(rec) if rec else 'none'}.
  Review-flagged.   {len(rev)} holding(s) REVIEW_REQUIRED — incomplete financial
                    data, routed to IC review: {', '.join(rev) if rev else 'none'}.
  EU Taxonomy.      {te}/{N} holdings have eligibility data, {ta}/{N} alignment —
                    present as an OVERLAY, not a portfolio compliance claim.
{gw_block}
  Watchlist.        {n_wl}/{N} holdings ({wl_w:.0f}% of weight) watchlisted — all
                    covered by the IC override decisions above.
{L}"""

print(factsheet)

fs_path = f"{REPORT_DIR}/factsheet_{TODAY}.txt"
with open(fs_path, "w", encoding="utf-8") as f:
    f.write(factsheet)
print(f"Factsheet saved: {fs_path}")


## ✅ Reporting complete

This is the last notebook in the pipeline. Outputs saved to `outputs/reports/`:
`portfolio_weights.png`, `esg_comparison.png`, `sector_allocation.png`, and `factsheet_<date>.txt` — plus the holdings, IC-reviewed and greenwashing 8-Test tables printed above.

**Pipeline run order** (see `CLAUDE.md` for the full agent table):

| # | Notebook | Agent | Status |
|---|----------|-------|--------|
| 01 | `01_mandate` | Mandate | ✅ |
| 02 | `02_data_ingestion` | Data Ingestion | ✅ |
| 03 | `03_data_quality` | Data Quality | ✅ |
| 06 | `06_document_intelligence` | Document Intelligence (RAG) | ✅ |
| 10 | `agent10_financial_analysis` | Financial Analysis | ✅ |
| 05 | `05_esg_climate` | ESG & Climate | ✅ |
| 07 | `07_biodiversity` | Biodiversity | ✅ |
| 08 | `08_eu_regulation` | EU Regulation | ✅ |
| 09 | `09_greenwashing` | Greenwashing 8-Test | ✅ |
| 10 | `10_portfolio_construction` | Portfolio Construction | ✅ |
| Opt | `Optimization_module/run_pipeline.py` | Portfolio Optimisation | ✅ |
| 11 | `11_human_review` | Human Review | ✅ |
| 12 | `12_reporting` | Reporting | ✅ *(this notebook)* |

**Remaining work (outside the notebooks):**
- n8n.cloud orchestration diagram for the report appendix (Pipeline Operator) — see `generate_pipeline_diagram.py`
- Written report sections, 5,000–6,500 words (all)
- Presentation deck + pre-recorded demo video
